# Covariance functions: from distance to matrix validity

This lab compares five Euclidean covariance shapes and checks the covariance matrix they generate for a small synthetic set of sites. The eigenvalue check is diagnostic, not a proof of universal validity.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd()
while root != root.parent and not (root / 'implementations').exists():
    root = root.parent
sys.path.insert(0, str(root / 'implementations' / 'python' / 'covariance-functions'))
from covariance import covariance_from_distance, covariance_matrix, covariance_matrix_diagnostics


## 1. Compare covariance shapes

The variance is held at one. The scale parameter is one, so the horizontal axis is distance divided by scale.

In [ ]:
h = np.linspace(0, 3, 250)
models = ['exponential', 'gaussian', 'matern32', 'spherical', 'wendland_c2']
for model in models:
    plt.plot(h, covariance_from_distance(h, model=model), label=model)
plt.xlabel('Euclidean distance h (range scale = 1)')
plt.ylabel('Covariance (variance = 1)')
plt.title('Five stationary covariance examples')
plt.legend()
plt.grid(True)
plt.show()


## 2. Build a covariance matrix

The sites are deliberately small and irregular. Inspect the matrix and its eigenvalue diagnostics.

In [ ]:
sites = np.array([[0.0, 0.0], [0.5, 0.2], [1.2, 0.1], [1.8, 0.9], [0.2, 1.5]])
for model in models:
    C = covariance_matrix(sites, model=model, variance=2.0, range_scale=0.8)
    print(model, covariance_matrix_diagnostics(C))


## 3. Explore range and duplicate locations

A larger range scale generally increases off-diagonal dependence. Duplicate locations create identical rows and columns, so the matrix can be singular while remaining positive semidefinite.

In [ ]:
for scale in (0.4, 0.8, 1.6):
    C = covariance_matrix(sites, model='exponential', range_scale=scale)
    print(f'range={scale}: covariance(site 0, site 1)={C[0, 1]:.4f}')

duplicate_sites = np.array([[0, 0], [0, 0], [1, 0]], dtype=float)
C_duplicate = covariance_matrix(duplicate_sites, model='gaussian')
print('duplicate-site matrix rank:', np.linalg.matrix_rank(C_duplicate), 'of', len(C_duplicate))
print(covariance_matrix_diagnostics(C_duplicate))


## 4. Take the result seriously

A numerical eigenvalue check is one diagnostic for one finite matrix. It does not establish that a formula is valid for every point set, dimension or geometry. The spherical and network cases need their own validity theory; see the linked method lesson before changing the distance metric.